In [ ]:

# Step 10–11 — Regression Test
# Check classification after the first Delta MERGE

from pyspark.sql import functions as F

valid_df = spark.table("default.rba_fx_v2_validated").select("source", "rate_date", "series_id", "rate")
master = spark.table("default.rba_fx_v2_master")

reclassified = (
    valid_df.alias("v")
    .join(
        master.alias("m"),
        (F.col("v.source") == F.col("m.source"))
        & (F.col("v.rate_date") == F.col("m.rate_date"))
        & (F.col("v.series_id") == F.col("m.series_id")),
        "left"
    )
    .select(
        "v.*",
        F.when(
            F.col("m.source").isNull(), "NEW"
        ).when(
            ~F.col("v.rate").eqNullSafe(F.col("m.rate")),
            "CHANGED"
        ).otherwise("UNCHANGED").alias("load_action")
    )
)

counts = {
    row["load_action"]: row["count"]
    for row in reclassified.groupBy("load_action").count().collect()
}

assert counts.get("NEW", 0) == 0
assert counts.get("CHANGED", 0) == 0
assert counts.get("UNCHANGED", 0) == valid_df.count()

print("Step 10–11 Regression Test: PASS")
print("NEW:", counts.get("NEW", 0))
print("CHANGED:", counts.get("CHANGED", 0))
print("UNCHANGED:", counts.get("UNCHANGED", 0))


In [ ]:

# Step 10–11 — CHANGED Regression Test
# Isolated test: does not modify the real RBA master

from pyspark.sql import functions as F

# Select one existing, valid FX record
original = (
    spark.table("default.rba_fx_v2_master")
    .orderBy("rate_date", "series_id")
    .limit(1)
)

assert original.count() == 1

# Create an independent Delta test table
original.write.format("delta").mode("overwrite").saveAsTable(
    "default.rba_fx_v2_changed_test"
)

# Simulate a source update: increase the rate by 0.01
incoming = (
    original
    .withColumn(
        "rate",
        (F.col("rate") + F.lit("0.01")).cast("decimal(20,8)")
    )
)

incoming.createOrReplaceTempView("changed_test_incoming")

# Classify against the test master
classification = (
    incoming.alias("s")
    .join(
        spark.table("default.rba_fx_v2_changed_test").alias("m"),
        ["source", "rate_date", "series_id"],
        "left"
    )
    .select(
        "s.*",
        F.when(
            F.col("m.source").isNull(), "NEW"
        ).when(
            ~F.col("s.rate").eqNullSafe(F.col("m.rate")),
            "CHANGED"
        ).otherwise("UNCHANGED").alias("load_action")
    )
)

display(classification)

assert classification.filter(
    F.col("load_action") == "CHANGED"
).count() == 1

print("CHANGED classification: PASS")


In [ ]:

# Step 10–11 — CHANGED MERGE Test
# Update the isolated test master, then verify UNCHANGED

from pyspark.sql import functions as F

# Capture the expected updated value
expected_rate = incoming.select("rate").first()["rate"]

# Reuse the incoming test data from the previous cell
spark.sql("""
MERGE INTO default.rba_fx_v2_changed_test AS target
USING changed_test_incoming AS source
ON  target.source = source.source
AND target.rate_date = source.rate_date
AND target.series_id = source.series_id

WHEN MATCHED AND NOT (target.rate <=> source.rate)
THEN UPDATE SET target.rate = source.rate

WHEN NOT MATCHED
THEN INSERT (source, rate_date, series_id, rate)
VALUES (
    source.source,
    source.rate_date,
    source.series_id,
    source.rate
)
""")

# Verify the updated Delta table
updated = spark.table("default.rba_fx_v2_changed_test")

assert updated.count() == 1
assert updated.select("rate").first()["rate"] == expected_rate

print("Delta MERGE update: PASS")

# Reclassify against the updated test master
after_merge = (
    incoming.alias("s")
    .join(
        updated.alias("m"),
        ["source", "rate_date", "series_id"],
        "left"
    )
    .select(
        "s.*",
        F.when(
            F.col("m.source").isNull(), "NEW"
        ).when(
            ~F.col("s.rate").eqNullSafe(F.col("m.rate")),
            "CHANGED"
        ).otherwise("UNCHANGED").alias("load_action")
    )
)

assert after_merge.filter(
    F.col("load_action") == "UNCHANGED"
).count() == 1

print("Post-MERGE classification: UNCHANGED")
print("CHANGED Regression Test: PASS")

display(after_merge)
